# LULC Classification: Uncertainty Quantification Pipeline
**Project:** Multispectral Image Analysis & Uncertainty Quantification  
**Author:** Danesh Selwal  
**Date:** 2026-05-02

---
## Executive Summary
This notebook evaluates post-hoc uncertainty over pretrained LULC classifiers and exports quantitative and spatial outputs for analysis.

**Objective:**
Apply uncertainty estimation methods, compare model behavior, and export reproducible artifacts to esults/ for reporting.

---
## 1. Environment Setup & Configuration
Import dependencies, configure runtime paths, and initialize reproducibility settings before running evaluation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

MODULE_NAME = 'ensemble'
REPO_ROOT = Path("/content/drive/MyDrive/Multispectral_Uncertainty_Quantification")
MODULE_DIR = REPO_ROOT / MODULE_NAME
RESULTS_DIR = MODULE_DIR / 'results'
MODELS_DIR = MODULE_DIR / 'models'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Module: {MODULE_NAME}')
print(f'Output Directory: {RESULTS_DIR}')


In [1]:
from pathlib import Path




Mounted at /content/drive


## 2. Data Ingestion & Preprocessing
Load multispectral inputs and reference labels, apply normalization, and prepare patch-based tensors for model training/evaluation.


In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow C++ INFO/WARNING logs
# ==============================================================================
# CELL 1: Initialization & Global Spatial Patch Extraction
# ==============================================================================
import os
import gc
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import tensorflow as tf
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)

# --- CONFIGURATION ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

MODEL_DIR = MODELS_DIR
DATA_DIR = REPO_ROOT / "data"
DATA_FILE = DATA_DIR / "data.csv"
LABEL_FILE = DATA_DIR / "ref.csv"

# Create Dedicated Subfolder for credal_ensemble Outputs
CREDAL_ENSEMBLE_OUT_DIR = RESULTS_DIR
CREDAL_ENSEMBLE_OUT_DIR.mkdir(parents=True, exist_ok=True)

PATCH_SIZE = 9

# --- DATA LOADING ---
print("Loading Multispectral Data...")

# -----------------------------
# Generalized Data Loading
# -----------------------------
import scipy.io as sio
import pandas as pd
import numpy as np

def universal_load_data(data_path, label_path):
    data_path = str(data_path)
    label_path = str(label_path)
    
    # Load features
    if data_path.endswith('.mat'):
        mat = sio.loadmat(data_path)
        x = next(v for k, v in mat.items() if not k.startswith('__') and isinstance(v, np.ndarray))
    elif data_path.endswith('.csv'):
        x = pd.read_csv(data_path).to_numpy(dtype=np.float32)
    elif data_path.endswith(('.tif', '.tiff')):
        try:
            import rasterio
            with rasterio.open(data_path) as src:
                x = src.read()
                x = np.moveaxis(x, 0, -1)
        except ImportError:
            print("rasterio not installed. Using dummy data.")
            x = np.zeros((10,10,3))

    # Load labels
    if label_path.endswith('.mat'):
        lmat = sio.loadmat(label_path)
        y = next(v for k, v in lmat.items() if not k.startswith('__') and isinstance(v, np.ndarray))
    elif label_path.endswith('.csv'):
        y = pd.read_csv(label_path).to_numpy(dtype=np.int32)
    elif label_path.endswith(('.tif', '.tiff')):
        try:
            import rasterio
            with rasterio.open(label_path) as src:
                y = src.read(1)
        except ImportError:
            y = np.zeros((10,10))

    # Normalization for 3D tensors
    if len(x.shape) == 3:
        x_norm = np.empty_like(x, dtype=np.float32)
        for b_idx in range(x.shape[-1]):
            band = x[:, :, b_idx]
            b_min, b_max = np.min(band), np.max(band)
            x_norm[:, :, b_idx] = (band - b_min) / max(b_max - b_min, 1e-8)
        x = x_norm
        
    return x, y

# Apply Generalized Loader
x_img, y_img = universal_load_data(DATA_FILE, LABEL_FILE)

# Handle flat CSVs by requesting user input or fallback
if len(x_img.shape) == 3:
    H, W, B = x_img.shape
else:
    print("WARNING: Data is flat. Please manually reshape x_img and y_img, then define H, W, B.")
    H, W, B = 330, 307, 6 # Default fallback for flat data

num_classes = int(np.unique(y_img).size)
print(f"Loaded Data Shape: {x_img.shape}, Labels Shape: {y_img.shape}, Classes: {num_classes}")

# Dynamic Color Palette Setup
import seaborn as sns
from matplotlib.colors import ListedColormap
BACKGROUND_COLOR = "#000000"
CLASS_COLOR_BASE = sns.color_palette("hls", max(10, num_classes)).as_hex()


TensorFlow: 2.19.0
Loading Multispectral Data...
Extracting all spatial patches into flat array for full-scene inference...
scene_pixels_scaled shape ready: (101310, 9, 9, 6)


In [3]:
# ==============================================================================
# CELL 2: Custom Keras Classes (Required for Deserialization)
# ==============================================================================
@tf.keras.utils.register_keras_serializable()
class PatchExtractor(layers.Layer):
    def __init__(self, patch_size=3, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
    def call(self, images):
        patches = tf.image.extract_patches(
            images=images, sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1], rates=[1, 1, 1, 1], padding="VALID"
        )
        batch, patch_dim = tf.shape(images)[0], tf.shape(patches)[-1]
        num_patches = tf.shape(patches)[1] * tf.shape(patches)[2]
        return tf.reshape(patches, [batch, num_patches, patch_dim])
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"patch_size": self.patch_size})
        return cfg

@tf.keras.utils.register_keras_serializable()
class PatchPositionEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)
    def call(self, patches):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patches) + self.position_embedding(positions)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return cfg

@tf.keras.utils.register_keras_serializable()
class GlobalFilterLayer(layers.Layer):
    def __init__(self, token_side, **kwargs):
        super().__init__(**kwargs)
        self.token_side = token_side
    def build(self, input_shape):
        channels = int(input_shape[-1])
        self.w_real = self.add_weight(name="w_real", shape=(self.token_side, self.token_side, channels), initializer="glorot_uniform", trainable=True)
        self.w_imag = self.add_weight(name="w_imag", shape=(self.token_side, self.token_side, channels), initializer="zeros", trainable=True)
        super().build(input_shape)
    def call(self, x):
        batch, channels = tf.shape(x)[0], tf.shape(x)[-1]
        x_2d = tf.reshape(x, [batch, self.token_side, self.token_side, channels])
        x_fft = tf.signal.fft2d(tf.cast(x_2d, tf.complex64))
        w_complex = tf.complex(self.w_real, self.w_imag)
        x_filtered = x_fft * w_complex
        x_spatial = tf.math.real(tf.signal.ifft2d(x_filtered))
        return tf.reshape(x_spatial, [batch, self.token_side * self.token_side, channels])
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"token_side": self.token_side})
        return cfg

@tf.keras.utils.register_keras_serializable()
class PatchEncoderWithCLS(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches + 1, output_dim=projection_dim)
    def build(self, input_shape):
        self.cls_token = self.add_weight(name="cls_token", shape=(1, 1, self.projection_dim), initializer="zeros", trainable=True)
        super().build(input_shape)
    def call(self, patches):
        batch = tf.shape(patches)[0]
        patch_proj = self.projection(patches)
        cls_tokens = tf.repeat(self.cls_token, repeats=batch, axis=0)
        x = tf.concat([cls_tokens, patch_proj], axis=1)
        positions = tf.range(start=0, limit=self.num_patches + 1, delta=1)
        return x + self.position_embedding(positions)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return cfg

# Bind them globally for load_model
CUSTOM_OBJECTS = {
    "PatchExtractor": PatchExtractor,
    "PatchPositionEncoder": PatchPositionEncoder,
    "GlobalFilterLayer": GlobalFilterLayer,
    "PatchEncoderWithCLS": PatchEncoderWithCLS,
}


In [4]:
# ==============================================================================
# CELL 3: Core Ensemble Evaluation Logic
# ==============================================================================
def get_ensemble_paths(model_name):
    search_pattern = str(MODEL_DIR / f"ensembles_{model_name}" / f"{model_name}_ens_*_final.keras")
    paths = glob.glob(search_pattern)
    if not paths: # Fallback
        search_pattern = str(MODEL_DIR / "ensembles_old" / f"{model_name}_ens_*_final.keras")
        paths = glob.glob(search_pattern)
    return paths

def evaluate_homogeneous_ensemble(model_paths, input_data, batch_size=2048):
    all_preds = []

    # Defensive RAM management
    for path in model_paths:
        print(f"  -> Loading & Predicting: {Path(path).name}")
        model = tf.keras.models.load_model(
            path, compile=False, custom_objects=CUSTOM_OBJECTS, safe_mode=False
        )
        preds = model.predict(input_data, batch_size=batch_size, verbose=1)
        all_preds.append(preds)

        # Obliterate the model from RAM immediately
        del model
        tf.keras.backend.clear_session()
        gc.collect()

    print("  -> Computing Credal Bounds...")
    stacked_preds = tf.stack(all_preds, axis=0) # Shape: (M, N_samples, C)
    p_min = tf.reduce_min(stacked_preds, axis=0)
    p_max = tf.reduce_max(stacked_preds, axis=0)

    delta_p = p_max - p_min
    p_star = p_min / (tf.reduce_sum(p_min, axis=-1, keepdims=True) + 1e-12)
    p_star = np.clip(p_star, 1e-12, 1.0)

    au = -np.sum(p_star * np.log(p_star), axis=-1)
    eu = np.mean(delta_p, axis=-1)
    tu = au + eu
    pred_class = np.argmax(p_star, axis=-1)

    return pred_class.numpy() if hasattr(pred_class, 'numpy') else pred_class, \
           p_star.numpy() if hasattr(p_star, 'numpy') else p_star, \
           au.numpy() if hasattr(au, 'numpy') else au, \
           eu.numpy() if hasattr(eu, 'numpy') else eu, \
           tu.numpy() if hasattr(tu, 'numpy') else tu


In [5]:
# ==============================================================================
# CELL 4: Standardized 6-Panel Spatial Mapping Engine
# ==============================================================================
def plot_and_save_credal_ensemble_masks(model_name, pred_class_scene, p_star_scene, au_scene, eu_scene, tu_scene, H=330, W=307, au_thresh=0.5, eu_thresh=0.2, tu_thresh=0.7):
    print(f"  -> Generating 6-Panel Spatial Maps...")
    n_classes = p_star_scene.shape[-1]

    # Reshape
    au_map = au_scene.reshape((H, W))
    eu_map = eu_scene.reshape((H, W))
    tu_map = tu_scene.reshape((H, W))
    pred_class_map = pred_class_scene.reshape((H, W))

    # Absolute Thresholding Masks (1 = Uncertain, 0 = Certain)
    au_mask = (au_map > au_thresh).astype(int)
    eu_mask = (eu_map > eu_thresh).astype(int)
    tu_mask = (tu_map > tu_thresh).astype(int)

    combined_au = np.where(au_mask == 1, n_classes, pred_class_map)
    combined_eu = np.where(eu_mask == 1, n_classes, pred_class_map)
    combined_tu = np.where(tu_mask == 1, n_classes, pred_class_map)

    # Palettes
#     CLASS_COLOR_BASE = ['#0000FF', '#00FF00', '#FF0000', '#00FFFF', '#FF00FF', '#FFFF00', '#A52A2A', '#FFA500', '#7FFF00', '#8A2BE2']
    cmap_classes = ListedColormap(CLASS_COLOR_BASE[:n_classes])
    cmap_with_unc = ListedColormap(CLASS_COLOR_BASE[:n_classes] + ['#808080'])
    cmap_binary = ListedColormap(['#FFFF00', '#001F3F'])

    fig, axes = plt.subplots(2, 3, figsize=(28, 16))
    fig.suptitle(f"{model_name} credal_ensemble Deep Ensemble Uncertainty (Absolute Cutoffs)", fontsize=22, fontweight='bold')

    # Panel 1: Base
    axes[0, 0].imshow(pred_class_map, cmap=cmap_classes, vmin=0, vmax=n_classes-1)
    axes[0, 0].set_title("Base Prediction Map (No Uncertainty)", fontsize=16)
    axes[0, 0].axis('off')

    # Panel 2: Binary TU
    axes[0, 1].imshow(tu_mask, cmap=cmap_binary, vmin=0, vmax=1)
    axes[0, 1].set_title(f"Certain vs Uncertain Map (Total Unc. > {tu_thresh})", fontsize=16)
    axes[0, 1].axis('off')
    axes[0, 1].legend(handles=[Patch(facecolor='#FFFF00', edgecolor='black', label='Certain'), Patch(facecolor='#001F3F', edgecolor='black', label='Uncertain')], loc='center left', bbox_to_anchor=(1.02, 0.5), framealpha=1.0, fontsize=14)

    # Panel 3: Bar Chart
    uniq, cnt = np.unique(combined_tu, return_counts=True)
    c_dict = {int(k): int(v) for k, v in zip(uniq, cnt)}
    vals = [c_dict.get(i, 0) for i in range(n_classes + 1)]
    axes[0, 2].bar([f"Class {i}" for i in range(n_classes)] + ["Uncertain"], vals, color=CLASS_COLOR_BASE[:n_classes] + ['#808080'], edgecolor='black')
    axes[0, 2].set_title(f"Pixel Counts per Class (TU > {tu_thresh})", fontsize=16)
    axes[0, 2].tick_params(axis='x', rotation=45, labelsize=12)
    for i, v in enumerate(vals): axes[0, 2].text(i, v + (max(vals) if vals else 1)*0.01, f'{v:,}', ha='center', va='bottom', fontweight='bold')

    # Panels 4-6: Grey Overlays
    axes[1, 0].imshow(combined_au, cmap=cmap_with_unc, vmin=0, vmax=n_classes)
    axes[1, 0].set_title(f"Aleatoric Mask Overlay (Grey AU > {au_thresh})", fontsize=16)
    axes[1, 0].axis('off')

    axes[1, 1].imshow(combined_eu, cmap=cmap_with_unc, vmin=0, vmax=n_classes)
    axes[1, 1].set_title(f"Epistemic Mask Overlay (Grey EU > {eu_thresh})", fontsize=16)
    axes[1, 1].axis('off')

    axes[1, 2].imshow(combined_tu, cmap=cmap_with_unc, vmin=0, vmax=n_classes)
    axes[1, 2].set_title(f"Total Mask Overlay (Grey TU > {tu_thresh})", fontsize=16)
    axes[1, 2].axis('off')

    plt.tight_layout(rect=[0, 0, 0.95, 1])

    # Save to the new subfolder
    save_path = CREDAL_ENSEMBLE_OUT_DIR / f"{model_name}_credal_ensemble_Spatial_Standardized.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"  -> Saved graphic to: {save_path}")


In [6]:
# ==============================================================================
# CELL 5: Master Execution & Memory Loop
# ==============================================================================
architectures_to_evaluate = ["AlexNet_CNN", "GFNet", "ViT_UNet"]
master_results = []

for model_name in architectures_to_evaluate:
    print(f"\n{'='*50}\nEvaluating credal_ensemble: {model_name}\n{'='*50}")

    ensemble_paths = get_ensemble_paths(model_name)
    if len(ensemble_paths) < 5:
        print(f"  [ERROR] Found only {len(ensemble_paths)} models for {model_name}. Skipping to next architecture.")
        continue

    # 1. Compute Inference & Mathematics
    pred_class, p_star, au, eu, tu = evaluate_homogeneous_ensemble(ensemble_paths, scene_pixels_scaled)

    # 2. Generate Apples-to-Apples Plot
    # Adjust thresholds here to match whatever you used in the CREDIT script!
    plot_and_save_credal_ensemble_masks(
        model_name, pred_class, p_star, au, eu, tu,
        H=H, W=W,
        au_thresh=0.5,
        eu_thresh=0.2,
        tu_thresh=0.7
    )

    # 3. Log Performance Metrics
    unique, counts = np.unique(pred_class, return_counts=True)
    pixel_counts = dict(zip(unique, counts))

    report_data = {
        "Model": f"{model_name}_credal_ensemble",
        "Mean_AU": float(np.mean(au)),
        "Mean_EU": float(np.mean(eu)),
        "Mean_TU": float(np.mean(tu)),
        **{f"Class_{int(k)}_Pixels": int(v) for k, v in pixel_counts.items()}
    }
    master_results.append(report_data)

    # 4. RUTHLESS MEMORY WIPE
    del pred_class, p_star, au, eu, tu
    tf.keras.backend.clear_session()
    gc.collect()

# Save Master Summary to the subfolder
if master_results:
    df_summary = pd.DataFrame(master_results)
    csv_path = CREDAL_ENSEMBLE_OUT_DIR / "credal_ensemble_master_summary.csv"
    df_summary.to_csv(csv_path, index=False)
    print(f"\n✅ All architectures evaluated. Master summary saved to: {csv_path}")
    print("\n--- credal_ensemble Master Summary ---")
    print(df_summary.to_string(index=False))


Output hidden; open in https://colab.research.google.com to view.